# Fine Tuning Base LLAMA as Chat [Instruct] Model


In [ ]:
import torch
from transformers import AutoTokenizer, pipeline
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model, AutoPeftModelForCausalLM
from transformers import TrainingArguments, Trainer
from trl import SFTTrainer, SFTConfig


In [ ]:
dataset = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft")
dataset = dataset.shuffle(seed=0).select(range(10_000))

In [ ]:
dataset[0]

In [ ]:
template_tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
template_tokenizer 

In [ ]:
def format_prompt(example):
  """Format the prompt using the <|user|> and <|assistant|> format"""
  chat = example['messages']
  prompt = template_tokenizer.apply_chat_template(chat, tokenize=False)

  return {'text': prompt}

In [ ]:
dataset = dataset.map(format_prompt)

In [ ]:
print(dataset[0]['text'])

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

## Model Configuration for Training

In [ ]:
# do the  4-bit quantization configuration in Q-LORA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype='float16',
    bnb_4bit_use_double_quant=True
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = "<PAD>"
tokenizer.padding_size="left"

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map = "auto",
    quantization_config=bnb_config
)

In [ ]:
model.config.use_cache=False
model.config.pretraining_tp=1

## Prepare LoRA Configuration for PEFT Fine tuning

In [ ]:
peft_config = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.1,
    r=64,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)

model = prepare_model_for_kbit_training(model)

model = get_peft_model(model, peft_config)

## Model Fine Tuning

In [ ]:
output_dir = "train_dir"

sft_args = SFTConfig(
    output_dir="train_dir",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    num_train_epochs=1,
    logging_steps=10,
    fp16=True,
    gradient_checkpointing=True,
    dataset_text_field="text",
    max_length=512,
)


trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_args,
    peft_config=peft_config,
    processing_class=tokenizer,  # tokenizer object in new API
)

trainer.train()

In [ ]:
trainer.model.save_pretrained("TinyLlama-1.1B-qlora")

## Load Pre-Trained PEFT Model for Prediction

In [ ]:
model = AutoPeftModelForCausalLM.from_pretrained(
    "TinyLlama-1.1B-qlora",
    device_map='auto'
)

merged_model = model.merge_and_unload()

In [ ]:
prompt = """<|user|>
Tell me something about Large Language Models.</s>
<|assistant|>
"""

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = "<PAD>"
tokenizer.padding_size="left"

pipe = pipeline(task='text-generation', model=merged_model, tokenizer=tokenizer)
output = pipe(prompt)
print(output[0]['generated_text'])

In [ ]:
!zip -r tiny_llama_qlora_adapter.zip TinyLlama-1.1B-qlora